<a href="https://colab.research.google.com/github/quinterogilalejandro-wq/MACHINE-LEARNING-/blob/main/Librer%C3%ADas_Numpy%2CPandas%2C_Sickit_Learn_Implementaci%C3%B3n_PARTE_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')


# ================================================================
# 1 — CARGA DE DATOS
# ================================================================
# ──  descarga directamente desde internet ──────────
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)


# ================================================================
#2.- Exploración de los datos (EDA). Determine valores nulos.
# ================================================================
print("\n" + "=" * 60)
print("P2 — EXPLORACIÓN DE DATOS (EDA)")
print("=" * 60)

print("\n--- Valores nulos por columna ---")
nulls = df.isnull().sum()
print(nulls)

print("\n2.1 — Variables CON valores nulos:")
vars_nulas = nulls[nulls > 0]
if len(vars_nulas) == 0:
    print("  No hay variables con valores nulos.")
else:
    for col, n in vars_nulas.items():
        pct = n / len(df) * 100
        print(f"  {col:12s}: {n:>4} nulos  ({pct:.1f}%)")

print("\n2.2 — Proporción de supervivencia (value_counts):")
print(df['Survived'].value_counts())
print()
print(df['Survived'].value_counts(normalize=True).round(4) * 100)
print("  → 0 = No sobrevivió  |  1 = Sobrevivió")


# ================================================================
# 3.- Elimine las columnas poco útiles.
# ================================================================
print("\n" + "=" * 60)
print("P3 — ELIMINACIÓN DE COLUMNAS POCO ÚTILES")
print("=" * 60)

cols_eliminar = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Embarked']
df.drop(columns=cols_eliminar, inplace=True)

print(f"Columnas eliminadas : {cols_eliminar}")
print(f"Columnas restantes  : {df.columns.tolist()}")
print(f"Nueva forma del df  : {df.shape}")


# ================================================================
# 4.- Defina un criterio para reemplazar los valores nulos.
# ================================================================
print("\n" + "=" * 60)
print("P4 — CRITERIO PARA VALORES NULOS")
print("=" * 60)

mediana_edad = df['Age'].median()
df['Age'].fillna(mediana_edad, inplace=True)

print(f"Variable 'Age': se rellenó con la MEDIANA = {mediana_edad} años")
print("Justificación: la mediana es más robusta que la media")
print("ante valores extremos (edades muy bajas o muy altas).")
print(f"\nNulos restantes en el dataset: {df.isnull().sum().sum()}")


# ================================================================
# 5.- Convierta la variable “Sex” a un valor numérico.
# ================================================================
print("\n" + "=" * 60)
print("P5 — CONVERSIÓN DE 'Sex' A NUMÉRICO")
print("=" * 60)

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

print("Mapeo aplicado: male → 0  |  female → 1")
print(df['Sex'].value_counts())


# ================================================================
# 6.- Lleve a cabo el entrenamiento del modelo utilizando “Regresión Logística”.
# ================================================================
print("\n" + "=" * 60)
print("P6 — ENTRENAMIENTO CON REGRESIÓN LOGÍSTICA")
print("=" * 60)

X = df.drop('Survived', axis=1)   # variables predictoras
y = df['Survived']                 # variable objetivo

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

print(f"Total de muestras   : {len(df)}")
print(f"Entrenamiento (80%) : {len(X_train)} muestras")
print(f"Prueba        (20%) : {len(X_test)} muestras")
print(f"\nVariables usadas    : {X.columns.tolist()}")
print(f"Modelo entrenado    ✅")


# ================================================================
# 7.- Haga la predicción del modelo (modelo.predict(X_test)).
# ================================================================
print("\n" + "=" * 60)
print("P7 — PREDICCIÓN (modelo.predict(X_test))")
print("=" * 60)

y_pred = modelo.predict(X_test)

print("Primeras 10 predicciones vs valores reales:")
comparacion = pd.DataFrame({
    'Real'    : y_test.values[:10],
    'Predicho': y_pred[:10],
    'Correcto': (y_test.values[:10] == y_pred[:10])
})
print(comparacion.to_string(index=False))


# ================================================================
# 8.- Proceda a realizar la evaluación del modelo. Utilizar una matriz de confusión.
# ================================================================
print("\n" + "=" * 60)
print("P8 — EVALUACIÓN DEL MODELO")
print("=" * 60)

cm  = confusion_matrix(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)

TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

print("\n         ┌─────────────────────────────────────────┐")
print("         │          MATRIZ DE CONFUSIÓN             │")
print("         ├──────────────────┬──────────────────────┤")
print("         │  Pred: No (0)    │  Pred: Sí (1)        │")
print("  ───────┼──────────────────┼──────────────────────┤")
print(f"  Real:0 │  TN = {TN:>4}       │  FP = {FP:>4}           │")
print(f"  Real:1 │  FN = {FN:>4}       │  TP = {TP:>4}           │")
print("         └──────────────────┴──────────────────────┘")
print(f"\n  TN (Verdadero Neg.): {TN}  → predijo No, era No  ✅")
print(f"  FP (Falso Pos.)    : {FP}  → predijo Sí, era No  ❌")
print(f"  FN (Falso Neg.)    : {FN}  → predijo No, era Sí  ❌")
print(f"  TP (Verdadero Pos.): {TP}  → predijo Sí, era Sí  ✅")
print(f"\nExactitud global (Accuracy): {acc:.4f}  ({acc*100:.1f}%)")
print("\nReporte completo de clasificación:")
print(classification_report(y_test, y_pred,
      target_names=['No sobrevivió (0)', 'Sobrevivió (1)']))


# ================================================================
# 9.- ¿Qué variables influyen más en la supervivencia?
# ================================================================
print("\n" + "=" * 60)
print("P9 — VARIABLES QUE MÁS INFLUYEN EN LA SUPERVIVENCIA")
print("=" * 60)

coeficientes = pd.DataFrame({
    'Variable'   : X.columns,
    'Coeficiente': modelo.coef_[0],
    'Abs'        : np.abs(modelo.coef_[0])
}).sort_values('Abs', ascending=False).drop(columns='Abs')

print(coeficientes.to_string(index=False))
print("\nInterpretación:")
print("  Coef. negativo → reduce probabilidad de sobrevivir.")
print("  Coef. positivo → aumenta probabilidad de sobrevivir.")
print("  Mayor valor absoluto → mayor influencia en el modelo.")


# ================================================================
# 10.- ¿El modelo detecta mejor sobrevivientes o fallecidos?
# ================================================================
print("\n" + "=" * 60)
print("P10 — ¿EL MODELO DETECTA MEJOR SOBREVIVIENTES O FALLECIDOS?")
print("=" * 60)

report_dict = classification_report(y_test, y_pred, output_dict=True)
recall_0 = report_dict['0']['recall']
recall_1 = report_dict['1']['recall']

print(f"Recall clase 0 (fallecidos)    : {recall_0:.4f}  →  {recall_0*100:.1f}% detectados correctamente")
print(f"Recall clase 1 (sobrevivientes): {recall_1:.4f}  →  {recall_1*100:.1f}% detectados correctamente")

mejor = "FALLECIDOS (clase 0)" if recall_0 > recall_1 else "SOBREVIVIENTES (clase 1)"
print(f"\n→ El modelo detecta mejor a los: {mejor}")
print("  El Recall mide qué tan bien el modelo identifica")
print("  los casos positivos reales de cada clase.")


# ================================================================
# 11.- Con base a los resultados de la Matriz de Confusión, ¿Es más grave un falso negativo o un falso positivo en este contexto?
# ================================================================
print("\n" + "=" * 60)
print("P11 — ¿MÁS GRAVE FALSO NEGATIVO O FALSO POSITIVO?")
print("=" * 60)

print(f"  Falsos Positivos  (FP = {FP}): predijo que sobrevivió, pero falleció.")
print(f"  Falsos Negativos  (FN = {FN}): predijo que falleció, pero sobrevivió.")
print()


# ================================================================
# 12.- Explique la instrucción “type” y aplíquela sobre el ejercicio.
# ================================================================
print("\n" + "=" * 60)
print("P12 — INSTRUCCIÓN type()")
print("=" * 60)
print("type() retorna la clase (tipo de dato) de cualquier objeto Python.")
print()
print(f"  type(df)              → {type(df)}")
print(f"  type(df['Age'])       → {type(df['Age'])}")
print(f"  type(df['Age'].iloc[0]) → {type(df['Age'].iloc[0])}")
print(f"  type(modelo)          → {type(modelo)}")
print(f"  type(42)              → {type(42)}")
print(f"  type('hola')          → {type('hola')}")
print(f"  type(3.14)            → {type(3.14)}")
print(f"  type([1, 2, 3])       → {type([1, 2, 3])}")


# ================================================================
# 13.- Explique la instrucción “tail” y aplíquela sobre el ejercicio.
# ================================================================
print("\n" + "=" * 60)
print("P13 — INSTRUCCIÓN tail()")
print("=" * 60)
print("tail(n) muestra las ÚLTIMAS n filas del DataFrame (default n=5).")
print("Útil para verificar el final del dataset o del resultado de un proceso.")
print()
print("df.tail()  →  últimas 5 filas:")
print(df.tail())
print()
print("df.tail(3)  →  últimas 3 filas:")
print(df.tail(3))


# ================================================================
# 14.- Explique la instrucción “unique” y aplíquela sobre el ejercicio.
# ================================================================
print("\n" + "=" * 60)
print("P14 — INSTRUCCIÓN unique()")
print("=" * 60)
print("unique() retorna los valores únicos de una Serie, sin repetición.")
print("Equivalente a un SELECT DISTINCT en SQL.")
print()
print(f"  df['Pclass'].unique()   → {df['Pclass'].unique()}")
print(f"  df['Sex'].unique()      → {df['Sex'].unique()}  (0=male, 1=female)")
print(f"  df['Survived'].unique() → {df['Survived'].unique()}  (0=No, 1=Sí)")
print(f"  df['SibSp'].unique()    → {sorted(df['SibSp'].unique())}")
print(f"  df['Parch'].unique()    → {sorted(df['Parch'].unique())}")


# ================================================================
# 15.- Aplique la función “describe( )” sobre la columna “edad” y explique los resultados obtenidos.
# ================================================================
print("\n" + "=" * 60)
print("P15 — describe() SOBRE COLUMNA 'Age'")
print("=" * 60)

desc = df['Age'].describe()
print(desc)
print()
print("Explicación de cada estadístico:")
print(f"  count  = {desc['count']:.0f}  → total de valores no nulos en la columna.")
print(f"  mean   = {desc['mean']:.2f}  → promedio de edad: ~{desc['mean']:.0f} años.")
print(f"  std    = {desc['std']:.2f}  → desviación estándar: qué tan dispersas están las edades.")
print(f"  min    = {desc['min']:.2f}  → edad mínima registrada.")
print(f"  25%    = {desc['25%']:.2f}  → el 25% de pasajeros tenía menos de {desc['25%']:.0f} años.")
print(f"  50%    = {desc['50%']:.2f}  → mediana: la mitad tenía menos de {desc['50%']:.0f} años.")
print(f"  75%    = {desc['75%']:.2f}  → el 75% tenía menos de {desc['75%']:.0f} años.")
print(f"  max    = {desc['max']:.2f}  → edad máxima registrada: {desc['max']:.0f} años.")


# ================================================================
# 16.- Multiplique todos los valores de la tarifa por 3.
# ================================================================
print("\n" + "=" * 60)
print("P16 — MULTIPLICAR TODOS LOS VALORES DE TARIFA POR 3")
print("=" * 60)

df['Fare_x3'] = df['Fare'] * 3

print("Columna 'Fare_x3' creada (Fare * 3).")
print()
print(df[['Fare', 'Fare_x3']].head(10).to_string(index=False))
print(f"\nSuma original  Fare   : {df['Fare'].sum():.2f}")
print(f"Suma nueva     Fare_x3: {df['Fare_x3'].sum():.2f}  (exactamente el triple ✅)")


# ================================================================
# 17.- Agrupe los datos por tarifa y contabilice las tarifas que sean igual a 71.
# ================================================================
print("\n" + "=" * 60)
print("P17 — ANÁLISIS DE SEGMENTACIÓN POR TARIFA (UMBRAL: 71 £)")
print("=" * 60)
grupo_fare = df.groupby('Fare').size().reset_index(name='Cantidad')
tarifas_71 = df[df['Fare'].astype(int) == 71]
print("Distribución de pasajeros por costo de pasaje (Top 10):")
print(grupo_fare.head(10).to_string(index=False))
print(f"\nTotal de registros identificados con tarifa ≈ 71: {len(tarifas_71)}")
print("Detalle de pasajeros segmentados:")
print(tarifas_71[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].to_string())



# ================================================================
# 18.- Agrupe los datos por edad y describa la cantidad de personas de cada edad.
# ================================================================
print("\n" + "=" * 60)
print("P18 — AGRUPAR POR EDAD Y CANTIDAD DE PERSONAS POR EDAD")
print("=" * 60)

grupo_edad = df.groupby('Age').size().reset_index(name='Cantidad')
grupo_edad = grupo_edad.sort_values('Age').reset_index(drop=True)

print(f"Total de edades distintas en el dataset: {len(grupo_edad)}")
print()
print("Cantidad de personas por cada edad:")
print(grupo_edad.to_string(index=False))


# ================================================================
# 19.- Agrupa los datos por edad y cuenta las edades que sean igual a 80.
# ================================================================
print("\n" + "=" * 60)
print("P19 — PERSONAS CON EDAD IGUAL A 80")
print("=" * 60)

grupo_edad_80 = df.groupby('Age').size().reset_index(name='Cantidad')
filtro_80 = grupo_edad_80[grupo_edad_80['Age'] == 80.0]

print("Resultado del groupby con edad == 80:")
print(filtro_80.to_string(index=False))

print("\nDetalle del/los pasajero(s) con edad 80:")
print(df[df['Age'] == 80.0][['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].to_string())






P2 — EXPLORACIÓN DE DATOS (EDA)

--- Valores nulos por columna ---
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

2.1 — Variables CON valores nulos:
  Age         :  177 nulos  (19.9%)
  Cabin       :  687 nulos  (77.1%)
  Embarked    :    2 nulos  (0.2%)

2.2 — Proporción de supervivencia (value_counts):
Survived
0    549
1    342
Name: count, dtype: int64

Survived
0    61.62
1    38.38
Name: proportion, dtype: float64
  → 0 = No sobrevivió  |  1 = Sobrevivió

P3 — ELIMINACIÓN DE COLUMNAS POCO ÚTILES
Columnas eliminadas : ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Embarked']
Columnas restantes  : ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
Nueva forma del df  : (891, 7)

P4 — CRITERIO PARA VALORES NULOS
Variable 'Age': se rellenó con la MEDIANA = 28.0 años
Justificación: 